In [ ]:
import pandas as pd
import numpy as np

import implicit
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
import hashlib

from scipy.sparse import coo_matrix
from implicit.evaluation import ranking_metrics_at_k

In [ ]:
df=pd.read_csv("Data/matriz.csv")

In [ ]:
# columna de codigos de categoria (para la matriz)
df["user_code"] = df["id_cliente_enc"].astype("category").cat.codes
df["item_code"] = df["cod_producto"].astype("category").cat.codes
df

In [ ]:
df.rename(columns={'id_cliente_enc': 'id'}, inplace=True)

In [ ]:
df['user_idx'] = df['user_code']
df['item_idx'] = df['item_code']

In [ ]:
matriz=df[["user_idx", "item_idx","cantidad"]]

Division train y test

In [ ]:
train=matriz.iloc[:int(0.8 * len(matriz))]
test=matriz.iloc[int(0.8 * len(matriz)):]

In [ ]:
#sparse matrix de interacciones usuario-ítem (orden correcto user item)    
def build_implicit_matrix(data):
    n_users = int(df['user_idx'].max() + 1)
    n_items = int(df['item_idx'].max() + 1)

    user_coords = data['user_idx'].values
    item_coords = data['item_idx'].values
    datos_matriz = np.ones(len(data))
    
    matrix = coo_matrix((datos_matriz, (user_coords, item_coords)), 
                        shape=(n_users, n_items))
    return matrix.tocsr()


train_matrix = build_implicit_matrix(train)
test_matrix = build_implicit_matrix(test)      

# asi las matrices tienen las dos el tamaño del dataset entero (sino da mal en ranking metrics)

El modelo vamos a hacerlo con menos factores porque con tantos, la segmentación no da resultados con sentido y la fiabilidad del modelo es muy baja. esto lo hemos decidido porque con 50 factores como teniamos, la clusterizacion era enorme y daba mucha dimensionalidad. con tantos factores la mejora no era "rentable" porque las metricas no mejoran de forma considerable

In [ ]:
# modelo y entrenamos con la matriz de train
model=AlternatingLeastSquares(factors=10, regularization=0.01, iterations=40, alpha= 20, random_state=42)
model.fit(train_matrix)

# CLUSTERING

CLUSTERING ITEMS --- KMEANS

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

In [ ]:
X = model.item_factors 

In [ ]:
scaler= StandardScaler()
data_scaled=scaler.fit_transform(X)

In [ ]:
#numero de componentes optimos
pca=PCA()
principal_components= pca.fit_transform(data_scaled)

cumulative_variance_ratio=np.cumsum(pca.explained_variance_ratio_) #se suman en cumsum porque nos interesa saber cuando parar
plt.plot(range(1,len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, marker="o")
plt.xlabel("Numero de Componentes Principales")
plt.ylabel("Varianza Explicada Acumulativa")
plt.title("Varianza Explicada Acumulativa vs. Numero de Componentes Principales")
plt.show()

In [ ]:
n_components=np.argmax(cumulative_variance_ratio >= 0.95) + 1
print(f"Numero óptimo de componentes principales: {n_components}")

In [ ]:
pca_final = PCA(n_components=n_components) 
principal_components = pca_final.fit_transform(data_scaled)

df_pca = pd.DataFrame(data=principal_components, columns=[f"PC{i}" for i in range(1, 11)])

In [ ]:
for col in df_pca.columns:
    inferior = df_pca[col].quantile(0.05)
    superior = df_pca[col].quantile(0.95)
    df_pca[col] = df_pca[col].clip(lower=inferior, upper=superior)

X_pca_limpio = df_pca.values

In [ ]:
dist = []
K = range(1,12) 
for k in K:
    kmeanModel = KMeans(n_clusters =k, random_state=42)
    kmeanModel.fit(X_pca_limpio)
    dist.append(kmeanModel.inertia_)

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(K,dist, marker="o")
plt.xlabel("Numero de clusters")
plt.ylabel("Inertia")
plt.title("Metodo del codo")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# kmeans óptimo
optimal_k=4
kmeans=KMeans(n_clusters=optimal_k, random_state=42)
kmeans.fit(X_pca_limpio)        #entreno con el que no tiene outliers y predigo con el que si
y_pred_i=kmeans.predict(X_pca_limpio)

In [ ]:
np.unique(y_pred_i)

In [ ]:
pd.Series(y_pred_i, name='cluster').value_counts().sort_index()

In [ ]:
# metrica silhouette para ver como de bien separados estan los clusters y si dentro los puntos están compactos
score= silhouette_score(df_pca,y_pred_i)
print(f"Silhouette score:{score:3f}")

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(X_pca_limpio[y_pred_i==0, 0], X_pca_limpio[y_pred_i==0, 1], s=100, c='red', label ='Cluster 0')
plt.scatter(X_pca_limpio[y_pred_i==1, 0], X_pca_limpio[y_pred_i==1, 1], s=100, c='blue', label ='Cluster 1')
plt.scatter(X_pca_limpio[y_pred_i==2, 0], X_pca_limpio[y_pred_i==2, 1], s=100, c='green', label ='Cluster 2')
plt.scatter(X_pca_limpio[y_pred_i==3, 0], X_pca_limpio[y_pred_i==3, 1], s=100, c='yellow', label ='Cluster 3')
plt.legend()
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Clusters sobre PCA')
plt.show() #los datos se distribuyen mas horizontalmente porque pc1 explica el mas varianza que pc2


CLUSTERING USERS --- KMEANS

In [ ]:
X = model.user_factors

In [ ]:
scaler= StandardScaler()
data_scaled=scaler.fit_transform(X)

In [ ]:
#numero de componentes optimos
pca=PCA()
principal_components= pca.fit_transform(data_scaled)

In [ ]:
cumulative_variance_ratio=np.cumsum(pca.explained_variance_ratio_) #se suman en cumsum porque nos interesa saber cuando parar
plt.plot(range(1,len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, marker="o")
plt.xlabel("Numero de Componentes Principales")
plt.ylabel("Varianza Explicada Acumulativa")
plt.title("Varianza Explicada Acumulativa vs. Numero de Componentes Principales")
plt.show()

In [ ]:
n_components=np.argmax(cumulative_variance_ratio >= 0.95) + 1
print(f"Numero óptimo de componentes principales: {n_components}")

In [ ]:
pca_final = PCA(n_components=n_components) 
principal_components = pca_final.fit_transform(data_scaled)

In [ ]:
df_pca = pd.DataFrame(data=principal_components, columns=[f"PC{i}" for i in range(1, 10)])

In [ ]:
for col in df_pca.columns:
    inferior = df_pca[col].quantile(0.05)
    superior = df_pca[col].quantile(0.95)
    df_pca[col] = df_pca[col].clip(lower=inferior, upper=superior)

X_pca_limpio = df_pca.values

In [ ]:
dist = []
K = range(1,11) 
for k in K:
    kmeanModel = KMeans(n_clusters =k, random_state=42)
    kmeanModel.fit(X_pca_limpio)
    dist.append(kmeanModel.inertia_)

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(K,dist, marker="o")
plt.xlabel("Numero de clusters")
plt.ylabel("Inertia")
plt.title("Metodo del codo")
plt.grid(True)
plt.show()

In [ ]:
optimal_k=3
kmeans=KMeans(n_clusters=optimal_k, random_state=42)
kmeans.fit(X_pca_limpio)        #entreno con el que no tiene outliers y predigo con el que si
y_pred=kmeans.predict(df_pca)

In [ ]:
np.unique(y_pred)

In [ ]:
pd.Series(y_pred, name='cluster').value_counts().sort_index()

In [ ]:
# metrica silhouette para ver como de bien separados estan los clusters y si dentro los puntos están compactos
score= silhouette_score(df_pca,y_pred)
print(f"Silhouette score:{score:3f}")

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(X_pca_limpio[y_pred==0, 0], X_pca_limpio[y_pred==0, 1], s=100, c='red', label ='Cluster 0')
plt.scatter(X_pca_limpio[y_pred==1, 0], X_pca_limpio[y_pred==1, 1], s=100, c='blue', label ='Cluster 1')
plt.scatter(X_pca_limpio[y_pred==2, 0], X_pca_limpio[y_pred==2, 1], s=100, c='green', label ='Cluster 2')
plt.legend()
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Clusters sobre PCA')
plt.show() #los datos se distribuyen mas horizontalmente porque pc1 explica el mas varianza que pc2

CLUSTERING ITEMS -- DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN
x = model.item_factors

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)  

In [ ]:
eps_values         = [1.0, 2.0, 3.0, 5.0, 8.0, 10.0, 15.0]
min_samples_values = [3, 5, 10, 15]

In [ ]:
results = []
for eps in eps_values:
    for ms in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=ms, metric='euclidean')
        labels = db.fit_predict(x_scaled)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_pct   = (labels == -1).sum() / len(labels) * 100
        valid = n_clusters >= 2 and noise_pct <= 50
        results.append({
            'eps': eps, 'min_samples': ms,
            'n_clusters': n_clusters,
            'noise_pct': round(noise_pct, 1),
            'valid': valid
        })

df_grid = pd.DataFrame(results)
print(df_grid)
df_best = df_grid[df_grid['valid']].sort_values(['n_clusters', 'noise_pct'], ascending=[False, True])
print(df_best.to_string(index=False))


In [ ]:
df_best= df_best[df_best["valid"]== True]
df_best

In [ ]:
dbscan_v1 = DBSCAN(eps=2.0, min_samples=5)
dbscan_v1.fit(x_scaled)

In [ ]:
labels = np.array(dbscan_v1.labels_)
print(labels)

In [ ]:
unicos = np.unique(labels)
print(unicos)

In [ ]:
pca=PCA(n_components=2, random_state=42).fit(x_scaled)
pca_2d=pca.transform(x_scaled)

In [ ]:
for i in range(0, pca_2d.shape[0]):
    if dbscan_v1.labels_[i] == 0:
        c1 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="r",       marker="+")
    elif dbscan_v1.labels_[i] == 1:
        c2 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="g",       marker="o")
    elif dbscan_v1.labels_[i] == 2:
        c3 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="orange",  marker="s")
    elif dbscan_v1.labels_[i] == 3:
        c4 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="purple",  marker="^")
    elif dbscan_v1.labels_[i] == 4:
        c5 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="cyan",    marker="D")
    elif dbscan_v1.labels_[i] == 5:
        c6 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="magenta", marker="v")
    elif dbscan_v1.labels_[i] == 6:
        c7 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="brown",   marker="x")
    elif dbscan_v1.labels_[i] == 7:
        c8 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="olive",   marker="p")
    elif dbscan_v1.labels_[i] == -1:
        c9 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="b",       marker="*")

plt.legend([c1, c2, c3, c4, c5, c6, c7, c8, c9],
           ["Cluster 1", "Cluster 2", "Cluster 3", "Cluster 4",
            "Cluster 5", "Cluster 6", "Cluster 7", "Cluster 8", "Noise"])
plt.title("DBSCAN - 8 Clusters")
plt.show()



In [ ]:
plt.scatter(pca_2d[dbscan_v1.labels_==0, 0], pca_2d[dbscan_v1.labels_==0, 1], c="r",        marker="+", label="Cluster 1")
plt.scatter(pca_2d[dbscan_v1.labels_==1, 0], pca_2d[dbscan_v1.labels_==1, 1], c="g",        marker="o", label="Cluster 2")
plt.scatter(pca_2d[dbscan_v1.labels_==2, 0], pca_2d[dbscan_v1.labels_==2, 1], c="orange",   marker="s", label="Cluster 3")
plt.scatter(pca_2d[dbscan_v1.labels_==3, 0], pca_2d[dbscan_v1.labels_==3, 1], c="purple",   marker="^", label="Cluster 4")
plt.scatter(pca_2d[dbscan_v1.labels_==4, 0], pca_2d[dbscan_v1.labels_==4, 1], c="cyan",     marker="D", label="Cluster 5")
plt.scatter(pca_2d[dbscan_v1.labels_==5, 0], pca_2d[dbscan_v1.labels_==5, 1], c="magenta",  marker="v", label="Cluster 6")
plt.scatter(pca_2d[dbscan_v1.labels_==6, 0], pca_2d[dbscan_v1.labels_==6, 1], c="brown",    marker="x", label="Cluster 7")
plt.scatter(pca_2d[dbscan_v1.labels_==7, 0], pca_2d[dbscan_v1.labels_==7, 1], c="olive",    marker="p", label="Cluster 8")
plt.scatter(pca_2d[dbscan_v1.labels_==-1, 0], pca_2d[dbscan_v1.labels_==-1, 1], c="b",     marker="*", alpha=0.3, label="Noise")
plt.legend()
plt.title("DBSCAN - 8 Clusters")
plt.show()

In [ ]:
print(dbscan_v1.labels_)
pd.Series(dbscan_v1.labels_).value_counts().sort_index().rename(index={-1: 'Noise'})

In [ ]:
print(dbscan_v1.labels_)
pd.Series(dbscan_v1.labels_).value_counts().sort_index().rename(index={-1: 'Noise'})

CLUSTERING USERS --- DBSCAN

In [ ]:
x = model.user_factors

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)  

In [ ]:
pca_pre = PCA(n_components=10, random_state=42)
x_reduced = pca_pre.fit_transform(x_scaled)


In [ ]:
x_sample = x_reduced[:35000]

In [ ]:
eps_values         = [1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
min_samples_values = [3, 5, 7, 10, 15]

results = []
for eps in eps_values:
    for ms in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=ms, metric='euclidean')
        labels = db.fit_predict(x_sample)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_pct   = (labels == -1).sum() / len(labels) * 100
        valid = n_clusters >= 2 and noise_pct <= 50
        results.append({
            'eps': eps, 'min_samples': ms,
            'n_clusters': n_clusters,
            'noise_pct': round(noise_pct, 1),
            'valid': valid
        })

df_grid = pd.DataFrame(results)
print(df_grid.to_string(index=False))

In [ ]:
df_grid= df_grid[df_grid["valid"]== True]
df_grid

In [ ]:
dbscan_v1 = DBSCAN(eps=1.5, min_samples=5)
dbscan_v1.fit(x_reduced)

In [ ]:
labels = np.array(dbscan_v1.labels_)
print(labels)

In [ ]:
print(dbscan_v1.labels_)
pd.Series(dbscan_v1.labels_).value_counts().sort_index().rename(index={-1: 'Noise'})

In [ ]:
for i in range(0, pca_2d.shape[0]):
    if dbscan_v1.labels_[i] == 0:
        c1 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="r", marker="+")
    elif dbscan_v1.labels_[i] == 1:
        c2 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="g", marker="o")
    elif dbscan_v1.labels_[i] == -1:
        c3 = plt.scatter(pca_2d[i,0], pca_2d[i,1], c="b", marker="*")

plt.legend([c1, c2, c3], ["Cluster 1", "Cluster 2", "Noise"])
plt.title("DBSCAN")
plt.show()

# SEGMENTACION

In [ ]:
df_users = df[['id', 'user_idx']].drop_duplicates()
df_users['cluster'] = y_pred 

df_interpretacion = df.merge(df_users[['user_idx', 'cluster']], on='user_idx')

df_interpretacion.groupby(['cluster', 'cod_producto'])['cantidad'].sum().reset_index()

In [ ]:
desc=pd.read_csv("Data/maestroestr.csv")

In [ ]:
desc.head()

In [ ]:
desc.rename(columns={'cod_est': 'cod_producto'}, inplace=True)

In [ ]:
top_products = (df_interpretacion
    .groupby(['cluster', 'cod_producto'])['cantidad']
    .sum()
    .reset_index()
    .sort_values(['cluster', 'cantidad'], ascending=[True, False])
)

top_products.groupby('cluster').head(15)

In [ ]:
print(top_products['cod_producto'].dtype)
print(desc['cod_producto'].dtype)

In [ ]:
top_products = pd.merge(top_products, desc, on='cod_producto', how='left')
top_products[['cluster', 'cod_producto', 'descripcion']].groupby('cluster').head(15)

ahora vemos que prodcutos compran los users de cada cluster. hay muchos productos que se repiten y claramente están los primeros, como el agua, el pan...

por ello hemos empezado a ver cuales eran los más comprados de cada cluster a partir de la posición 75, que es donde hemos visto productos más personales

In [ ]:
top_products_filtrado = top_products.sort_values(['cluster', 'cantidad'], ascending=[True, False])

top_products_filtrado = top_products_filtrado.groupby('cluster').apply(lambda x: x.iloc[75:]
).reset_index(drop=True)

top_products_filtrado[['cluster', 'cod_producto', 'descripcion']].groupby('cluster').head(10)

ver ticket de un cliente del cluster, a ver que compra

In [ ]:
df_interpretacion = df_interpretacion.merge(desc, on='cod_producto', how='left')

In [ ]:
cliente = df_interpretacion[df_interpretacion['cluster'] == 0]['user_idx'].iloc[0]

df_interpretacion[df_interpretacion['user_idx'] == cliente][['cod_producto', 'descripcion', 'cantidad']]

In [ ]:

cliente2= df_interpretacion[df_interpretacion['cluster'] == 1]['user_idx'].iloc[1]
df_interpretacion[df_interpretacion['user_idx'] == cliente2][['cod_producto', 'descripcion', 'cantidad']]

In [ ]:
cliente3 = df_interpretacion[df_interpretacion['cluster'] == 2]['user_idx'].iloc[0]

df_interpretacion[df_interpretacion['user_idx'] == cliente3][['cod_producto', 'descripcion', 'cantidad']]

In [ ]:
lista = [4637,43465,43536,4984,24587,28386,9301,20490,7112,5349]
df_users[df_users['user_idx'].isin(lista)][['user_idx', 'cluster']]

ITEM DEL OBJETIVO 1 EN QUE CLUSTER ESTÁ

In [ ]:
df_items = df[['cod_producto']].drop_duplicates().copy()
df_items['cluster'] = y_pred_i

df_items

In [ ]:
nombre = pd.merge(df_items, desc, on='cod_producto', how='left')
nombre[['cluster', 'cod_producto', 'descripcion']].groupby('cluster').head(15)

In [ ]:
producto = nombre[nombre['cod_producto'] == 4201510]
producto

el producto que habia que promocionar a 10 clientes está en el cluster 1 de items. ahora vamos a mirar que tipo de productos hay en cada cluster

In [ ]:
nombre[nombre["cluster"]==0].head(50)

In [ ]:
nombre[nombre["cluster"]==1].head(50)

In [ ]:
nombre[nombre["cluster"]==2].head(50)

In [ ]:
nombre[nombre["cluster"]==3].head(50)

Como se puede ver el item promocionado que son taquitos de bacon, comparte cluster con productos bastante similares, como masa de pizza, chorizo, tipos de queso, panceta...